In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


In [15]:
spark = SparkSession.builder \
 .appName("OptimizationAndCachingDemo") \
 .getOrCreate()

In [4]:
print("Generating 10 million rows...")
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)

Generating 10 million rows...


26/06/15 03:07:16 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
optimized_df = df.filter(col("value") > 500).filter(col("id") <
5000000).select("id", "value")

In [6]:
print("Physical Execution Plan (Notice how it pushes filters down):")
optimized_df.explain()

Physical Execution Plan (Notice how it pushes filters down):
== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [7]:
print("\n=========================================")
print("PART B: CACHING EXECUTION TIMES")
print("=========================================")
start_time = time.time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time.time()
print(f"1. UNCACHED Execution | Count: {count_uncached} | Time:{round(end_time - start_time, 4)} seconds")



PART B: CACHING EXECUTION TIMES


[Stage 0:>                                                          (0 + 2) / 2]

1. UNCACHED Execution | Count: 2495000 | Time:1.9069 seconds


In [8]:
print("\n--> Applying .cache() (This is lazy, nothing happens yet)...")


--> Applying .cache() (This is lazy, nothing happens yet)...


In [9]:
optimized_df.cache()


DataFrame[id: bigint, value: bigint]

In [11]:
start_time = time.time()
count_first_cache = optimized_df.count() # Action triggers caching
end_time = time.time()
print(f"2. FIRST CACHED Action | Count: {count_first_cache} | Time:{round(end_time - start_time, 4)} seconds (Materializing cache)")

[Stage 3:=============================>                             (1 + 1) / 2]

2. FIRST CACHED Action | Count: 2495000 | Time:2.8192 seconds (Materializing cache)


In [13]:
start_time = time.time()
count_second_cache = optimized_df.count()
end_time = time.time()
print(f"3. SECOND CACHED Action| Count: {count_second_cache} | Time:{round(end_time - start_time, 4)} seconds (Read from memory)")


3. SECOND CACHED Action| Count: 2495000 | Time:0.1668 seconds (Read from memory)


In [14]:
optimized_df.unpersist()
print("\n--> Unpersisted DataFrame to free up memory.")
spark.stop()


--> Unpersisted DataFrame to free up memory.
